# 01 · The Sparrow Model Zoo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/01_model_zoo.ipynb)

The zoo lists every model Sparrow-Engine knows about: camera-trap detectors and
classifiers, overhead (drone and aerial) models, bioacoustic models and image encoders.
This notebook browses the catalog, checks licences, and installs a model.

**Licences vary per model.** Some are AGPL-3.0, some are non-commercial (CC-BY-NC).
Check the licence column before you use a model.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
import pandas as pd

zoo = spe.zoo_models()
df = pd.DataFrame(
    [(m.id, m.domain, m.task, m.license, m.downloadable, m.default) for m in zoo],
    columns=["id", "domain", "task", "license", "downloadable", "default"],
)
print(len(df), "models")
df.groupby(["domain", "task"]).size().rename("models").to_frame()

Camera-trap detectors you can download:

In [ ]:
df[(df.domain == "camera_trap") & (df.task == "detector") & df.downloadable]

Models by licence. `downloadable == False` means the model is listed but not hosted, usually because its licence is unverified.

In [ ]:
pd.crosstab(df.license, df.downloadable)

## Install a model

`download_model` fetches the zip from Zenodo, verifies its MD5 and installs it. It skips models that are already installed.

In [ ]:
path = spe.download_model("MDV6-yolov10-c")
path

In [ ]:
installed = pd.DataFrame(
    [(m.id, m.model_type, m.default) for m in spe.list_models()],
    columns=["id", "model_type", "default"],
)
installed.sort_values(["model_type", "id"])

In [ ]:
info = spe.model_info("MDV6-yolov10-c")
print(info)
print(spe.verify_model("MDV6-yolov10-c"))

## More ways to install

```python
spe.download_models(["MDV6-yolov10-e", "AI4G-Amazon-V2"])   # several at once
spe.download_models(["bioclip-2"], dest="/data/models")      # custom directory
```

From a shell, use `python scripts/download_models.py <id> [<id> ...]` in this repository.
Set `SPARROW_ENGINE_MODEL_DIR` to use a model directory other than `~/.sparrow-engine/models`.